In [ ]:
import json, os
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    DataCollatorForLanguageModeling, Trainer, TrainingArguments
)

# ---- Config ----
MODEL_NAME = "google/reformer-crime-and-punishment"  # character-level Reformer
DATA_DIR = "data"
BLOCK_SIZE = 2048  # long contexts are fine for Reformer
OUTPUT_DIR = "reformer-chatbot-checkpoints"

# ---- Load tokenizer & model ----
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

# Reformer checkpoints above are char-level; they often lack pad_token.
# Make sure pad is defined for batching.
if tok.pad_token is None:
    tok.add_special_tokens({"pad_token": "<|pad|>"})

SPECIAL_TOKENS = {"additional_special_tokens": ["<|user|>", "<|assistant|>"]}
tok.add_special_tokens(SPECIAL_TOKENS)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tok))  # after adding tokens
model.config.pad_token_id = tok.pad_token_id
model.config.bos_token_id = tok.bos_token_id if tok.bos_token_id else tok.eos_token_id
model.config.eos_token_id = tok.eos_token_id

# ---- Load data ----
ds = load_dataset("json", data_files={
    "train": os.path.join(DATA_DIR, "train.jsonl"),
    "validation": os.path.join(DATA_DIR, "valid.jsonl")
})

# ---- Tokenize ----
def tok_fn(ex):
    out = tok(ex["text"], truncation=True, max_length=BLOCK_SIZE)
    return out

tokenized = ds.map(tok_fn, batched=False, remove_columns=ds["train"].column_names)

# ---- Causal LM labels = inputs shifted inside the model; no MLM ----
collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)

# ---- Train ----
args = TrainingArguments(
    OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    evaluation_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_steps=1000,
    num_train_epochs=2,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=500,
    fp16=True,
    bf16=False,
    lr_scheduler_type="cosine",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=collator,
    tokenizer=tok
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)
